### Installation

In [ ]:
%%capture
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN_READ")

In [ ]:
OUTPUT_DIR = "/kaggle/working/output/qwen-2.5-7b-instruct"
REPO_ID = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
os.makedirs(OUTPUT_DIR, exist_ok=True)

### Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 512
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = REPO_ID,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = hf_token,
    device_map = "balanced",
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = True, # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("HizkiaJ/qed_sermon_100k", token=hf_token)
train_ds = dataset['train']
dev_ds = dataset['validation'] 

print(f"Final Train Data: {len(train_ds)}")
print(f"Final Validation Data: {len(dev_ds)}")

In [ ]:
import time
import json
import os
from transformers import TrainerCallback

class EpochTimeCallback(TrainerCallback):
    def __init__(self, log_filepath):
        self.log_filepath = log_filepath
        self.epoch_start_time = None

    def on_epoch_begin(self, args, state, control, **kwargs):
        self.epoch_start_time = time.time()

    def on_epoch_end(self, args, state, control, **kwargs):
        if self.epoch_start_time is not None:
            epoch_duration_seconds = time.time() - self.epoch_start_time
            epoch_duration_minutes = round(epoch_duration_seconds / 60, 2)
            
            log_data = {
                "epoch": round(state.epoch, 2),
                "duration_seconds": round(epoch_duration_seconds, 2),
                "duration_minutes": epoch_duration_minutes
            }
            
            with open(self.log_filepath, "a") as f:
                f.write(json.dumps(log_data) + "\n")
                
            print(f"\n[INFO] Epoch execution time {round(state.epoch, 2)}: {epoch_duration_minutes} minutes.")

class StopAfterOneEpochCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        print(f"\n[INFO] Epoch {round(state.epoch, 2)} is done.")
        control.should_training_stop = True

In [ ]:
def formatting_func(example):
    messages = [
        {"role": "system", "content": "You are a professional translator. Translate the following Indonesian text to English. Provide only the translation, without any explanations or additional text."},
        {"role": "user", "content": example['text_id']},
        {"role": "assistant", "content": example['text_en']}
    ]
    
    text = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=False,
    )
    return {"text": text}

train_ds = train_ds.map(formatting_func)
dev_ds = dev_ds.map(formatting_func)

train_ds = train_ds.remove_columns(["text_en", "text_id"])
dev_ds = dev_ds.remove_columns(["text_en", "text_id"])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

LOG_FILE = f"{OUTPUT_DIR}/training_time.jsonl"
time_logger = EpochTimeCallback(log_filepath=LOG_FILE)
early_stopping = EarlyStoppingCallback(early_stopping_patience=1)
stop_trigger = StopAfterOneEpochCallback()

args = TrainingArguments(
    output_dir = OUTPUT_DIR,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 32,
    warmup_steps = 25,
    num_train_epochs = 3,
    learning_rate = 2e-5,
    logging_steps = 5,
    eval_strategy="epoch",
    save_strategy="epoch",
    optim = "paged_adamw_8bit",
    greater_is_better=False,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    metric_for_best_model="eval_loss",
    weight_decay = 0.01,
    save_total_limit=1,
    lr_scheduler_type = "constant",
    report_to = "none",
    seed = 3407,
    load_best_model_at_end=True
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = dev_ds,
    args = args,
    dataset_text_field = "text",
    max_seq_length = 512,
    packing = False,
    callbacks=[early_stopping, time_logger, stop_trigger],
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
# CHECKPOINT_PATH = ""
# trainer_stats = trainer.train(resume_from_checkpoint=CHECKPOINT_PATH)
trainer_stats = trainer.train()

In [ ]:
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)